In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.4 Embeddings

A token id is just a number; the model needs a vector. An embedding table is a
lookup: row `i` is the learned vector for token `i`. Training shapes those rows.

In [ ]:
from torch import nn

vocab, dim = 50, 8
emb = nn.Embedding(vocab, dim)
ids = torch.tensor([1, 2, 3])
vecs = emb(ids)
print("ids", ids.tolist(), "-> vectors of shape", tuple(vecs.shape))

The same id always maps to the same row, so identical tokens start identical
and only diverge through context in later layers.

In [ ]:
assert tuple(vecs.shape) == (3, 8)
assert torch.equal(emb(torch.tensor([1]))[0], emb.weight[1])